In [9]:
import pickle

import numpy as np
import torch
import torch.nn as nn
from pydantic import BaseModel

from torch.utils.data import TensorDataset, DataLoader
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split

import wandb
import copy

In [16]:
wandb.login()
rng = np.random.default_rng()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/hutchii/.netrc.
wandb: Currently logged in as: hutchii (hutchtech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
class FLClient(BaseModel):
    id: str
    model: dict = None
    num_samples: int = None

class AffectModel(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(in_features=6, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=128),
            nn.ReLU(),
            nn.Linear(in_features=128, out_features=64),
            nn.ReLU(),
            nn.Linear(in_features=64, out_features=num_classes),
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
#subjects = ["S6","S7","S8","S9"]
subjects = ["S7","S8","S9"]

DATA_DIR = "/Users/hutchii/projects/MentalHealthLLM/data/wesad_subjects"

# WESAD readme: label 0 is transient/undefined and 5-7 are "to be ignored".
# Only these four are real study conditions. Values are remapped to 0..3.
LABEL_MAP = {1: 0, 2: 1, 3: 2, 4: 3}
LABEL_NAMES = ["baseline", "stress", "amusement", "meditation"]
NUM_CLASSES = len(LABEL_MAP)

num_clients = 3 # number of clients that participate in each round
num_rounds = 5
BATCH_SIZE = 256
MAX_EPOCH = 10
LEARNING_RATE = 1e-3
PROJECT_PREFIX = "stress-fl"

TEST_SIZE = 0.25
BLOCK_SIZE = 7000  # 10s @ 700Hz -- the unit that train/test is split on, see block_split()
SPLIT_SEED = 42    # fixed so the held-out set is identical across rounds and sessions
EVAL_BATCH = 8192

torch.manual_seed(0)
agg_model = AffectModel(num_classes=NUM_CLASSES)

In [ ]:
#Data Processing

def init_fl_participants(subjects: list[str], n: int) -> dict[str, FLClient]:
    '''
    Takes a list of subject IDs as input and returns a dictionary of FLClients
    :param n: number of random subjects returned
    :param subjects: total list of subject ID's
    :return: dict[str, FLClient]: A list of randomly selected ID's
    '''
    participants = dict()
    selected =  rng.choice(subjects, size=n, replace=False)
    print(selected)

    for p_id in selected:
        client = FLClient(id=p_id)
        participants[p_id] = client

    return participants

def get_subject_samples(data:dict) -> torch.Tensor:
    '''
    Converts raw subject data into tensor to be used for training
    :param data: dictionary containing raw subject data
    :return: torch.Tensor
    '''
    chest_signals = data['signal']['chest']

    c_acc = chest_signals['ACC']
    c_acc_np = np.array(c_acc)
    acc_mag = np.sum(c_acc_np**2, axis=1)

    acc = acc_mag.reshape(-1, 1)
    ecg = chest_signals['ECG']
    emg = chest_signals['EMG']
    eda = chest_signals['EDA']
    temp = chest_signals['Temp']
    resp = chest_signals['Resp']

    return torch.tensor(np.stack([acc, ecg, emg, eda, temp, resp], axis=1).squeeze()).float()

def get_subject_targets(data:dict) -> torch.Tensor:
    '''
    Converts raw subject data into targets to be used during training
    :param data: Dictionary containing raw data
    :return: torch.Tensor
    '''
    targets = data['label']
    return torch.tensor(targets.reshape(-1, 1)).long().squeeze(dim=1)

def filter_study_labels(samples: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    '''
    Drops the WESAD labels that are not study conditions (0 transient, 5-7 ignore) and
    remaps the remaining ones to a contiguous 0..NUM_CLASSES-1 range.
    :param samples: signal tensor, shape (N, 6)
    :param targets: raw WESAD labels 0..7, shape (N,)
    :return: (filtered samples, remapped targets)
    '''
    keep = torch.isin(targets, torch.tensor(list(LABEL_MAP.keys())))
    samples, targets = samples[keep], targets[keep]

    remapped = torch.empty_like(targets)
    for raw, new in LABEL_MAP.items():
        remapped[targets == raw] = new

    return samples, remapped

def block_split(samples: torch.Tensor, targets: torch.Tensor) -> tuple[tuple, tuple]:
    '''
    Splits into train/test by contiguous blocks rather than individual samples.
    At 700Hz, neighbouring samples are near-identical, so a per-sample shuffled split puts
    almost every test sample next to a twin in train and the accuracy it reports is fiction.
    Whole blocks go to one side or the other, stratified on each block's majority label.
    :param samples: signal tensor, shape (N, 6)
    :param targets: remapped targets, shape (N,)
    :return: ((x_train, y_train), (x_test, y_test))
    '''
    n = len(targets)
    n_blocks = int(np.ceil(n / BLOCK_SIZE))
    block_ids = np.arange(n_blocks)
    block_labels = np.array([
        torch.mode(targets[b * BLOCK_SIZE:(b + 1) * BLOCK_SIZE]).values.item()
        for b in block_ids
    ])

    train_b, test_b = train_test_split(
        block_ids,
        test_size=TEST_SIZE,
        stratify=block_labels,
        shuffle=True,
        random_state=SPLIT_SEED,
    )

    def gather(blocks):
        idx = np.concatenate([
            np.arange(b * BLOCK_SIZE, min((b + 1) * BLOCK_SIZE, n)) for b in np.sort(blocks)
        ])
        return samples[idx], targets[idx]

    return gather(train_b), gather(test_b)

_subject_splits: dict[str, dict] = {}

def get_subject_split(subject: str) -> dict:
    '''
    Loads a subject, filters to the study conditions, splits by block and normalizes using
    statistics from that subject's *train* split only. Cached, because the pickles are ~830MB
    and because the held-out set has to stay fixed across rounds.
    :param subject: The subject ID
    :return: dict with x_train / y_train / x_test / y_test
    '''
    if subject in _subject_splits:
        return _subject_splits[subject]

    with open(f"{DATA_DIR}/{subject}.pkl", 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    s_samples = get_subject_samples(data)
    s_targets = get_subject_targets(data)
    s_samples, s_targets = filter_study_labels(s_samples, s_targets)

    (x_train, y_train), (x_test, y_test) = block_split(s_samples, s_targets)

    # mean/std come from train only, then get applied to both sides
    x_train_norm, x_test_norm = z_score_norm(x_train, x_test)

    print(f"{subject}: {len(y_train)} train / {len(y_test)} test samples")

    _subject_splits[subject] = {
        "x_train": x_train_norm, "y_train": y_train,
        "x_test": x_test_norm, "y_test": y_test,
    }
    return _subject_splits[subject]

def get_data_loader(subject:str, batch_size:int) -> DataLoader:
    '''
    Given a subject, return a DataLoader over its training split.
    :param subject: The subject ID
    :return: DataLoader
    '''
    split = get_subject_split(subject)
    ds = TensorDataset(split["x_train"], split["y_train"])
    return DataLoader(dataset=ds, shuffle=True, batch_size=batch_size)

def z_score_norm(data: torch.Tensor, test_samples: torch.Tensor = None):
    '''
    Performs Z score normalization for provided data. If test samples also provided normaization occurs there as well.
    :param data: Total population of data to be standardized, the mean and STD will be calculated from this
    :param test_samples: Split of data to be normalized using the same mu and std
    :return: normalized data, or (normalized data, normalized test samples)
    '''
    mean = data.mean(dim=0, keepdim=True)
    std = data.std(dim=0, keepdim=True)

    samples = (data - mean) / std

    if test_samples is not None:
        test_samples = (test_samples - mean) / std
        return samples, test_samples

    return samples

#Models
def train(model, subject, wab_run, dataloader, e, loss_fn, optim):
    for e in tqdm(range(e), desc='Epochs'):
        for x, y in tqdm(dataloader, desc='Batch'):
            model.train()
            pred = model(x)
            loss = loss_fn(pred, y)

            wab_run.log({
                f"{subject}_training_loss":loss.item()
            })

            optim.zero_grad()
            loss.backward()
            optim.step()

    return model.state_dict()

def fed_avg(clients:dict[str, FLClient], total_samples) -> dict:
    layers = dict()
    all_models = []
    sample_lens = []

    for k,v in clients.items():
        all_models.append(v.model)
        sample_lens.append(v.num_samples)

    model_keys = all_models[0].keys()

    for k in model_keys:
        shape = all_models[0][k].shape
        weighted_sum = torch.zeros(shape)
        for i, m in enumerate(all_models):
            weight = sample_lens[i] / total_samples
            weighted_sum += weight * m[k]
        layers[k] = weighted_sum

    return layers

# Eval
def evaluate_model(m: nn.Module, x: torch.Tensor, y: torch.Tensor) -> tuple[float, float]:
    '''
    Runs the model over x in batches and scores it against y.
    Balanced accuracy (mean per-class recall) is reported alongside plain accuracy because
    the conditions are imbalanced -- baseline is roughly 3x amusement.
    :return: (accuracy %, balanced accuracy %)
    '''
    m.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(x), EVAL_BATCH):
            preds.append(torch.argmax(m(x[i:i + EVAL_BATCH]), dim=1))
    pred = torch.cat(preds)

    accuracy = (pred == y).float().mean().item() * 100
    recalls = [
        (pred[y == c] == c).float().mean().item()
        for c in range(NUM_CLASSES) if (y == c).any()
    ]
    balanced = float(np.mean(recalls)) * 100

    return accuracy, balanced

def majority_baseline(y: torch.Tensor) -> float:
    '''
    Accuracy of always predicting the most common class -- the floor any real result has to clear.
    '''
    return torch.unique(y, return_counts=True)[1].max().item() / len(y) * 100

def fed_avg_accuracy(layers: dict, subjects:list[str], wab_run=None, rnd:int=None) -> dict:
    '''
    Scores the aggregated model on each subject's HELD-OUT split.
    :param layers: aggregated state_dict
    :param subjects: subjects to evaluate on
    :return: dict of per-subject (accuracy, balanced accuracy)
    '''
    m = AffectModel(num_classes=NUM_CLASSES)
    m.load_state_dict(layers)

    results = {}
    for s in subjects:
        split = get_subject_split(s)
        x, y = split["x_test"], split["y_test"]

        accuracy, balanced = evaluate_model(m, x, y)
        results[s] = (accuracy, balanced)

        print(f"{s} test accuracy {accuracy:.2f}% | balanced {balanced:.2f}% "
              f"(majority baseline {majority_baseline(y):.2f}%)")

        if wab_run is not None:
            wab_run.log({
                f"{s}_test_accuracy": accuracy,
                f"{s}_test_balanced_accuracy": balanced,
                "round": rnd,
            })

    mean_acc = float(np.mean([a for a, _ in results.values()]))
    mean_bal = float(np.mean([b for _, b in results.values()]))
    print(f"round mean: accuracy {mean_acc:.2f}% | balanced {mean_bal:.2f}%")

    if wab_run is not None:
        wab_run.log({
            "global_test_accuracy": mean_acc,
            "global_test_balanced_accuracy": mean_bal,
            "round": rnd,
        })

    return results

def get_model_accuracy(logits: torch.Tensor, targets:torch.Tensor) -> float:
    probs = torch.softmax(logits, dim=1)
    ind = torch.argmax(probs, dim=1)

    diff = (ind == targets).int()
    correct = torch.sum(diff, dim=0).item()
    accuracy =  float(format((correct / len(logits)) * 100, ".2f"))
    return accuracy


In [41]:
run = wandb.init(
    entity='hutchtech',
    project="stress-fl",
    name=f"{PROJECT_PREFIX}-{rng.random()}",
    config={
        "lr":LEARNING_RATE,
        "epochs":MAX_EPOCH,
        "batch_size":BATCH_SIZE
    }
)

wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /Users/hutchii/projects/MentalHealthLLM/notebooks/wandb/run-20260817_225747-idwm9jsy
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run stress-fl-0.18286913922799075
wandb: ⭐️ View project at https://wandb.ai/hutchtech/stress-fl
wandb: 🚀 View run at https://wandb.ai/hutchtech/stress-fl/runs/idwm9jsy


In [ ]:
loss_fn = nn.CrossEntropyLoss()

for n in range(num_rounds):
    print(f"--- round {n} ---")
    # reset every round so the FedAvg weights sum to 1
    total_samples = 0
    clients = init_fl_participants(subjects, num_clients)

    for k,v in clients.items():
        dl = get_data_loader(k, BATCH_SIZE)

        num_samples = len(dl.dataset)
        total_samples += num_samples

        model = copy.deepcopy(agg_model)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

        client_model = train(
            model=model,
            subject=k,
            wab_run=run,
            dataloader=dl,
            e=MAX_EPOCH,
            loss_fn=loss_fn,
            optim=optimizer
        )

        updated_client = FLClient(
            id=k,
            model=client_model,
            num_samples=num_samples
        )
        clients[k] = updated_client

    # keep agg_model a module across rounds; fed_avg returns a state_dict
    layers = fed_avg(clients, total_samples=total_samples)
    agg_model.load_state_dict(layers)

    fed_avg_accuracy(layers, subjects, wab_run=run, rnd=n)

In [159]:
run.finish()